# Assignment 04: Blood Pressure Follow-Up

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python3 check_assignment.py` in the terminal.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)

DATA_PATH = Path("data") / "bp_followup.csv"
HOME_PATH = Path("data") / "home_bp.csv"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
LOADED_PATH = OUTPUT_DIR / "bp_loaded.csv"
SUMMARY_PATH = OUTPUT_DIR / "visit_summary.csv"
COUNTS_PATH = OUTPUT_DIR / "clinic_counts.csv"
FOLLOWUP_PATH = OUTPUT_DIR / "followup_priority.csv"
PARQUET_PATH = OUTPUT_DIR / "followup_priority.parquet"
GAP_PATH = OUTPUT_DIR / "white_coat_gap.csv"

print("data files found:", DATA_PATH.exists() and HOME_PATH.exists())

ModuleNotFoundError: No module named 'numpy'

## Task 1: Put the cells in running order

### 1.1 Reorder the cells

The next two code cells are out of order: the first uses `vials_on_hand`, which the second defines. Move the whole producer cell above the dependent cell. Do not copy the definition into another cell.

In [ ]:
vials_on_hand = 3
print("vials_on_hand:", vials_on_hand)

In [ ]:
doses_available = vials_on_hand * 10
print("doses_available:", doses_available)

### 1.2 Explain the repair

Explain the difference between the order the cells appear in and the order they ran, why the first cell can print a result even while it sits above the cell it needs, why output saved under a cell is not proof that the notebook works now, and what you changed and how Restart and Run All shows that it worked.

To begin, "periwinkle." 

The difference between the order the cells appear in and the order they ran:
The cells appear in a fixed visual order in the notebook, but the kernel runs cells in whatever order I execute them. This means the execution order can be different from the order shown on the page.

Why the first cell can print a result even while it sits above the cell it needs:
The first cell can print a result if vials_on_hand was already defined in the kernel from an earlier run. The kernel remembers variables until it is restarted, even if the cell defining the variable is literally below the cell using it.

Why output saved under a cell is not proof that the notebook works now: Output svaed under a cell is not proof that the notework works now because it only shows that the cell successfully ran at some point in the past. It does not prove that the notebook can run correctly from a fresh kernel in its current cell order.

What I changed and how Restart and Run All shows that it worked:
I moved the cell that defines vials_on_hand above the cell that uses it to calculate doses_available. Restart clears the kernel's old variables, and Run All executes the notebook from top to bottom. Since it then prints vials_on_hand: 3 and doses_available: 30 without an error, the new order works correctly. 

## Task 2: Load the blood pressure export

### 2.1 Look at the raw file

Run this cell as it is: it prints the file and reads it with no options.

In [ ]:
print(open(DATA_PATH).read())
raw = pd.read_csv(DATA_PATH)
print("shape:", raw.shape)
raw.head(3)

### 2.2 Read with options and save

> **Checkpoint: `output/bp_loaded.csv`**
> 14 rows under the header `patient_id,clinic,age,sbp_baseline,sbp_week4,sbp_week8`.

In [ ]:
bp = pd.read_csv(
    DATA_PATH,
    sep=";",
    skiprows=[1],
    usecols=[
        "patient_id",
        "clinic",
        "age",
        "sbp_baseline",
        "sbp_week4",
        "sbp_week8",
    ],
    na_values=["-999"],
    index_col="patient_id",
)

print("shape:", bp.shape)
print(bp.dtypes)
display(bp.head())

display(bp.loc["P104"])
display(bp.iloc[0:3])

bp.to_csv(LOADED_PATH)

## Task 3: Summarize the readings

### 3.1 Summarize each visit

> **Checkpoint: `output/visit_summary.csv`**
> Three rows, `sbp_baseline`, `sbp_week4`, and `sbp_week8`, under the header `visit,mean,median,count`.

In [ ]:
readings = bp[["sbp_baseline", "sbp_week4", "sbp_week8"]]

visit_summary = pd.DataFrame({
    "mean": readings.mean(),
    "median": readings.median(),
    "count": readings.count(),
})

visit_summary.index.name = "visit"

display(visit_summary)

highest_baseline = bp["sbp_baseline"].idxmax()

baseline_week8_r = bp["sbp_baseline"].corr(
    bp["sbp_week8"]
)
print("highest baseline:", highest_baseline)
print("baseline vs week 8 r:", baseline_week8_r)

visit_summary.to_csv(SUMMARY_PATH)

### 3.2 Count patients per clinic

> **Checkpoint: `output/clinic_counts.csv`**
> Four rows under the header `clinic,count`, North first.

In [ ]:
clinic_counts = bp["clinic"].value_counts()
display(clinic_counts)
print("clinics:", bp["clinic"].nunique())

clinic_counts.to_csv(COUNTS_PATH)

## Task 4: Build the follow-up list

### 4.1 Select the program patients

In [ ]:
in_program = (
    bp["clinic"].isin(["North", "East"])
    & (bp["sbp_baseline"] >= 140)
)
followup = bp.loc[in_program].drop(
    columns=["age"]
)

print("program patients:", in_program.sum())

followup

### 4.2 Add the derived columns

In [ ]:
followup["sbp_mean"] = followup[
    ["sbp_baseline", "sbp_week4", "sbp_week8"]
].mean(axis="columns")

followup["change_week8"] = (
    followup["sbp_week8"]
    - followup["sbp_baseline"]
)

followup

### 4.3 Sort, rank, and save

> **Checkpoint: `output/followup_priority.csv` and `output/followup_priority.parquet`**
> Seven patients, P110 first and P107 last, with the columns `README.md` lists.

In [ ]:
followup = followup.sort_values(
    by=["change_week8", "patient_id"]
)

followup["improvement_rank"] = (
    followup["change_week8"].rank(method="min")
)
display(followup)

followup.to_csv(FOLLOWUP_PATH)
followup.to_parquet(PARQUET_PATH)

back = pd.read_parquet(PARQUET_PATH)
print(
    "Parquet round trip matches:",
    back.equals(followup)
)

## Task 5: Compare clinic and home readings

> **Checkpoint: `output/white_coat_gap.csv`**
> 15 rows under the header `patient_id,white_coat_gap_mmhg`, five of them with a number.

In [ ]:
home = pd.read_csv(
    HOME_PATH,
    index_col="patient_id"
)

white_coat_gap = (
    bp["sbp_week8"]
    - home["home_sbp_week8"]
)

white_coat_gap.name = "white_coat_gap_mmhg"

display(white_coat_gap)
print(
    "patients with both readings:",
    white_coat_gap.count()
)

white_coat_gap.to_csv(GAP_PATH)

## Fresh-run check

Click **Restart**, then **Run All**. The last cell prints your score and what to fix.

In [ ]:
from check_assignment import run_checks

_ = run_checks()

: 